# Analisis Pengujian VueJS dan Svelte

Notebook awal untuk mengolah hasil pengujian berpasangan. Setiap baris data mewakili satu blok eksperimen untuk metrik dan skenario yang sama.

In [ ]:
from pathlib import Path

import matplotlib
import pandas as pd
import scipy
import statsmodels

print(f"pandas {pd.__version__}")
print(f"scipy {scipy.__version__}")
print(f"statsmodels {statsmodels.__version__}")
print(f"matplotlib {matplotlib.__version__}")

## Memuat data

Simpan data utama di `data/lighthouse_runs.csv` dengan kolom `block_id`, `scenario`, `metric`, `vue`, dan `svelte`. Nilai mentah VueJS dan Svelte harus dipasangkan dalam kondisi eksperimen yang sebanding.

In [ ]:
DATA_PATH = Path("data/lighthouse_runs.csv")
COLUMNS = ["block_id", "scenario", "metric", "vue", "svelte"]

data = pd.read_csv(DATA_PATH) if DATA_PATH.exists() else pd.DataFrame(columns=COLUMNS)
missing_columns = set(COLUMNS) - set(data.columns)
if missing_columns:
    raise ValueError(f"Kolom wajib tidak tersedia: {sorted(missing_columns)}")

if data.duplicated(["block_id", "scenario", "metric"]).any():
    raise ValueError("Setiap block_id, scenario, dan metric harus unik.")

data["vue"] = pd.to_numeric(data["vue"])
data["svelte"] = pd.to_numeric(data["svelte"])
data["d_i"] = data["vue"] - data["svelte"]
data

## Ringkasan deskriptif

Untuk FCP, LCP, Speed Index, TBT, dan CLS, nilai yang lebih rendah lebih baik. Karena itu, `d_i > 0` berarti nilai VueJS lebih tinggi daripada Svelte pada pasangan tersebut.

In [ ]:
summary = (
    data.groupby(["scenario", "metric"])[["vue", "svelte", "d_i"]]
    .agg(["count", "mean", "std", "median", "min", "max"])
)
summary

## Analisis lanjutan

Gunakan selanjutnya untuk pemeriksaan Q–Q plot dan Shapiro–Wilk pada `d_i`, lalu pilih *paired t-test*, Wilcoxon, atau uji tanda/permutasi sesuai karakteristik selisih. Terapkan koreksi Holm pada keluarga lima metrik Lighthouse.

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests